# ModelObsCompare — model vs obs (Phase 2)

Compare an **exported ISSM model** against observed velocities: spatial
difference maps over an ROI, modelled flowline overlays, and paired time series.

**Prerequisite:** export the model once in MATLAB (see
`examples/example_export_model.m`):

```matlab
M = moc_load_model('Model_NW_HindcastRun_TransientInversion.mat', ...
                   'solution','transient','trange',[2018 2020]);
moc_export_model(M, 'exported/Model_NW_2018_2020.nc', 'fields',{'vel','vx','vy'});
```

The Python side reads that netCDF with `h5py` (no ISSM needed) and interpolates
the mesh with `matplotlib.tri` — the same P1 linear interpolation as ISSM's
`InterpFromMesh2d` / `InterpFromMeshToGrid`.

In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt
import moc_py as mp

# point to your exported model netCDF (bare name resolves against CONFIG.model_dir)
MODEL_NC = "exported/Model_NW_fric1_Snapshot.nc"
M = mp.load_model(MODEL_NC)
print("mesh:", M.x.size, "vertices,", M.elements.shape[0], "elements | time:", M.time)

## 1. Load obs + define ROI

In [ ]:
O = mp.load_obs_netcdf("Sentinel_Subset_Upernavik.nc", bands=["vv","ex","ey"],
                       trange=(2018, 2020))
roi = mp.roi_from_obs(O, margin=2000)      # or mp.load_roi("upernavik_box")

## 2. Spatial difference map (model - obs)

In [ ]:
D = mp.spatial_diff(M, O, roi, field="vel", obsband="vv",
                    res=mp.CONFIG.grid_res, time=2019.6, obswindow=0.1)
mp.plot_diff(D, roi=roi)
plt.show()
print(f"RMSE={D.attrs['rmse']:.0f}  MAE={D.attrs['mae']:.0f}  "
      f"bias={D.attrs['bias']:.0f} m/yr over {D.attrs['n']} px")

## 3. Flowline: model overlaid on observations

In [ ]:
F  = mp.load_obs_flowline("jakobshavn.mat")      # observed flowline
Fm = mp.model_flowline(M, F, field="vel")        # model sampled along the SAME line

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
mp.plot_flowline(F, mode="profile", time=2019.0, model=Fm, ax=axs[0])
mp.plot_flowline(F, mode="hovmoller", model=Fm, ax=axs[1])   # model-obs difference
plt.tight_layout(); plt.show()

## 4. Paired time series at a point

In [ ]:
C = mp.compare_timeseries(M, F, dist=5000, field="vel")   # 5 km up-glacier
mp.plot_compare(C)
plt.show()
print(f"n={C.stats['n']}  RMSE={C.stats['rmse']:.0f}  bias={C.stats['mean']:.0f}  "
      f"r={C.stats.get('r', float('nan')):.2f}")

# ... or straight from the gridded product (nearest pixel):
# C2 = mp.compare_timeseries(M, O, xy=(float(F.x[20]), float(F.y[20])), obsband="vv")
# mp.plot_compare(C2); plt.show()

## Validation

The Python model path is validated end-to-end (no MATLAB needed) by
`tests/test_model_synthetic.py`: a known linear field must interpolate exactly
and a model-vs-itself difference must be ~0. It also checks robustness to the
stored axis order. Run: `python tests/test_model_synthetic.py`.